# Fidelity Estimation Deep Dive

This notebook explores `FidelityEstimator`, the component that predicts
whether a circuit will produce useful results on a given backend *before*
you spend money running it.

We cover:
- Full circuit fidelity estimation with `estimate()`
- Quick planning estimates with `estimate_depth_limited_fidelity()`
- Error source breakdown: gate errors, readout errors, decoherence
- Fidelity scaling with circuit depth and 2Q gate count
- Comparing predicted fidelity across backends
- When to trust (and distrust) fidelity estimates

In [1]:
from qb_compiler.noise.fidelity_estimator import FidelityEstimator
from qb_compiler.noise.fidelity_estimator import QBCircuit as FECircuit
from qb_compiler.noise.empirical_model import EmpiricalNoiseModel
from qb_compiler.calibration.static_provider import StaticCalibrationProvider
from qb_compiler.config import BACKEND_CONFIGS

## 1. Basic Fidelity Estimation

`FidelityEstimator.estimate()` takes a circuit and a noise model, then
computes the probability that a single shot returns the correct output.
It combines three error sources multiplicatively:

1. **Gate fidelity product** -- `(1 - error)` for every gate
2. **Decoherence penalty** -- T1/T2 relaxation over total gate time per qubit
3. **Readout penalty** -- `(1 - readout_error)` for every measured qubit

In [2]:
# Load real calibration data and build a noise model
provider = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)
noise_model = EmpiricalNoiseModel.from_calibration(provider)
print(f"Backend: {provider.backend_name}")
print(f"Noise model: {noise_model}")

Backend: ibm_fez
Noise model: EmpiricalNoiseModel(n_qubits=156, n_gates=352)


In [3]:
# Build a simple 3-qubit GHZ circuit using the FidelityEstimator's circuit type
ghz_circuit = FECircuit(
    gates=[
        ("h", (0,)),
        ("cx", (0, 1)),
        ("cx", (1, 2)),
    ],
    n_qubits=3,
    measurements=frozenset({0, 1, 2}),
)

estimator = FidelityEstimator()
fidelity = estimator.estimate(ghz_circuit, noise_model)
print(f"Estimated fidelity for 3-qubit GHZ: {fidelity:.4f}")
print(f"Interpretation: ~{fidelity*100:.1f}% of shots expected to be correct")

Estimated fidelity for 3-qubit GHZ: 0.8907
Interpretation: ~89.1% of shots expected to be correct


## 2. Quick Estimates with `estimate_depth_limited_fidelity()`

When you don't have a full circuit yet -- just a rough gate count -- use
this method for rapid planning. It only accounts for gate errors
(no decoherence or readout), so it gives an **upper bound** on fidelity.

In [4]:
estimator = FidelityEstimator()

# Planning: "I need ~50 CX gates and ~100 single-qubit gates.
#            IBM Fez has median CX error ~0.5%, 1Q error ~0.1%.
#            Will it work?"

quick_fidelity = estimator.estimate_depth_limited_fidelity(
    n_two_qubit_gates=50,
    avg_two_qubit_error=0.005,
    n_one_qubit_gates=100,
    avg_one_qubit_error=0.001,
)
print(f"Quick estimate (gate errors only): {quick_fidelity:.4f}")
print(f"Note: actual fidelity will be lower due to decoherence + readout")

Quick estimate (gate errors only): 0.7042
Note: actual fidelity will be lower due to decoherence + readout


## 3. Error Source Breakdown

Let's isolate each error source to understand which dominates for a
given circuit. We do this by selectively zeroing out error channels.

In [5]:
from qb_compiler.calibration.models.qubit_properties import QubitProperties
from qb_compiler.calibration.models.coupling_properties import GateProperties


def build_noise_model(gate_err, readout_err, t1_us, t2_us, n_qubits=5):
    """Build a uniform noise model with controllable error rates."""
    qprops = {
        q: QubitProperties(
            qubit_id=q, t1_us=t1_us, t2_us=t2_us,
            readout_error=readout_err, frequency_ghz=5.0,
        )
        for q in range(n_qubits)
    }
    gprops = {}
    for q in range(n_qubits):
        gprops[("sx", (q,))] = GateProperties("sx", (q,), gate_err * 0.1, 40.0)
        gprops[("h", (q,))] = GateProperties("h", (q,), gate_err * 0.1, 40.0)
    for q in range(n_qubits - 1):
        gprops[("cx", (q, q + 1))] = GateProperties(
            "cx", (q, q + 1), gate_err, 400.0,
        )
    return EmpiricalNoiseModel(qubit_props=qprops, gate_props=gprops)


# 10-qubit chain circuit: H on q0, then 9 CNOT gates
n = 5
chain_gates = [("h", (0,))] + [("cx", (i, i + 1)) for i in range(n - 1)]
chain = FECircuit(gates=chain_gates, n_qubits=n, measurements=frozenset(range(n)))

est = FidelityEstimator()

# Full error model
nm_full = build_noise_model(0.005, 0.01, 300.0, 150.0, n)
f_full = est.estimate(chain, nm_full)

# Gate errors only (perfect readout, infinite coherence)
nm_gates_only = build_noise_model(0.005, 0.0, 1e9, 1e9, n)
f_gates = est.estimate(chain, nm_gates_only)

# Readout errors only (perfect gates, infinite coherence)
nm_readout_only = build_noise_model(0.0, 0.01, 1e9, 1e9, n)
f_readout = est.estimate(chain, nm_readout_only)

# Decoherence only (perfect gates, perfect readout)
nm_decoherence = build_noise_model(0.0, 0.0, 300.0, 150.0, n)
f_decoherence = est.estimate(chain, nm_decoherence)

print(f"Error source breakdown for {n}-qubit chain circuit:")
print(f"  Full model fidelity:      {f_full:.4f}")
print(f"  Gate errors only:         {f_gates:.4f}  (loss: {1-f_gates:.4f})")
print(f"  Readout errors only:      {f_readout:.4f}  (loss: {1-f_readout:.4f})")
print(f"  Decoherence only:         {f_decoherence:.4f}  (loss: {1-f_decoherence:.4f})")

Error source breakdown for 5-qubit chain circuit:
  Full model fidelity:      0.9019
  Gate errors only:         0.9797  (loss: 0.0203)
  Readout errors only:      0.9510  (loss: 0.0490)
  Decoherence only:         0.9681  (loss: 0.0319)


## 4. Fidelity Scaling with Circuit Depth

Fidelity decays exponentially with the number of noisy operations.
Let's plot how fidelity drops as we increase the number of 2Q gates.

In [6]:
est = FidelityEstimator()

# Vary 2Q gate count and compute fidelity for different error rates
gate_counts = list(range(0, 201, 5))
error_rates = [0.001, 0.005, 0.01, 0.02]

print(f"{'2Q Gates':>8} | ", end="")
for err in error_rates:
    print(f" err={err:<6}", end="")
print()
print("-" * 55)

for n_gates in [0, 10, 25, 50, 75, 100, 150, 200]:
    print(f"{n_gates:>8} | ", end="")
    for err in error_rates:
        f = est.estimate_depth_limited_fidelity(
            n_two_qubit_gates=n_gates,
            avg_two_qubit_error=err,
        )
        print(f"  {f:.4f}  ", end="")
    print()

2Q Gates |  err=0.001  err=0.005  err=0.01   err=0.02  
-------------------------------------------------------
       0 |   1.0000    1.0000    1.0000    1.0000  
      10 |   0.9900    0.9511    0.9044    0.8171  
      25 |   0.9753    0.8822    0.7778    0.6035  
      50 |   0.9512    0.7783    0.6050    0.3642  
      75 |   0.9277    0.6866    0.4706    0.2198  
     100 |   0.9048    0.6058    0.3660    0.1326  
     150 |   0.8606    0.4715    0.2215    0.0483  
     200 |   0.8186    0.3670    0.1340    0.0176  


## 5. Impact of Different Gate Types

Two-qubit gates (CX, CZ, ECR) have error rates ~10x higher than
single-qubit gates. Let's quantify the fidelity impact of each type.

In [7]:
est = FidelityEstimator()
n_qubits = 5
nm = build_noise_model(0.005, 0.01, 300.0, 150.0, n_qubits)

# Circuit with only 1Q gates (10 Hadamards)
only_1q = FECircuit(
    gates=[("h", (q,)) for q in range(n_qubits)] * 2,
    n_qubits=n_qubits,
    measurements=frozenset(range(n_qubits)),
)

# Circuit with 1Q + 2Q gates (GHZ-like)
mixed_gates = [("h", (0,))] + [("cx", (i, i+1)) for i in range(n_qubits - 1)]
mixed = FECircuit(
    gates=mixed_gates,
    n_qubits=n_qubits,
    measurements=frozenset(range(n_qubits)),
)

# Circuit heavy on 2Q gates (4 layers of CX)
heavy_2q_gates = []
for layer in range(4):
    for i in range(0, n_qubits - 1, 2 if layer % 2 == 0 else 1):
        if i + 1 < n_qubits:
            heavy_2q_gates.append(("cx", (i, i + 1)))
heavy_2q = FECircuit(
    gates=heavy_2q_gates,
    n_qubits=n_qubits,
    measurements=frozenset(range(n_qubits)),
)

print("Gate type impact on fidelity:")
print(f"  10x 1Q gates only:   {est.estimate(only_1q, nm):.4f}")
print(f"  GHZ ({n_qubits-1} CX + 1 H):  {est.estimate(mixed, nm):.4f}")
print(f"  Heavy 2Q ({len(heavy_2q_gates)} CX):  {est.estimate(heavy_2q, nm):.4f}")

Gate type impact on fidelity:
  10x 1Q gates only:   0.9425
  GHZ (4 CX + 1 H):  0.9019
  Heavy 2Q (12 CX):  0.8135


## 6. Comparing Predicted Fidelity Across Backends

Each backend has different error rates, coherence times, and gate durations.
Let's compare expected fidelity for the same circuit across all 9 backends
using their median specs from `BACKEND_CONFIGS`.

In [8]:
est = FidelityEstimator()

# Test circuit: 5-qubit GHZ (representative of a shallow entangling circuit)
test_circuit = FECircuit(
    gates=[
        ("h", (0,)),
        ("cx", (0, 1)),
        ("cx", (1, 2)),
        ("cx", (2, 3)),
        ("cx", (3, 4)),
    ],
    n_qubits=5,
    measurements=frozenset(range(5)),
)

print(f"{'Backend':<18} {'CX err':>8} {'RO err':>8} {'T2 (us)':>8} {'Fidelity':>10}")
print("-" * 56)

results = []
for name, spec in sorted(BACKEND_CONFIGS.items()):
    # Build a uniform noise model from the backend's median specs
    nm = build_noise_model(
        gate_err=spec.median_cx_error,
        readout_err=spec.median_readout_error,
        t1_us=spec.t1_us,
        t2_us=spec.t2_us,
        n_qubits=5,
    )
    f = est.estimate(test_circuit, nm)
    results.append((name, f))
    print(
        f"{name:<18} {spec.median_cx_error:>8.4f} "
        f"{spec.median_readout_error:>8.4f} {spec.t2_us:>8.0f} {f:>10.4f}"
    )

best = max(results, key=lambda x: x[1])
worst = min(results, key=lambda x: x[1])
print(f"\nBest:  {best[0]} ({best[1]:.4f})")
print(f"Worst: {worst[0]} ({worst[1]:.4f})")

Backend              CX err   RO err  T2 (us)   Fidelity
--------------------------------------------------------
ibm_fez              0.0050   0.0100      150     0.9019
ibm_marrakesh        0.0055   0.0110      140     0.8938
ibm_torino           0.0060   0.0120      130     0.8856
ionq_aria            0.0040   0.0030   500000     0.9690
ionq_forte           0.0030   0.0030   500000     0.9730
iqm_emerald          0.0080   0.0150       20     0.7036
iqm_garnet           0.0150   0.0200       15     0.6145
quantinuum_h2        0.0010   0.0020  1000000     0.9860
rigetti_ankaa        0.0200   0.0300       10     0.4862

Best:  quantinuum_h2 (0.9860)
Worst: rigetti_ankaa (0.4862)


## 7. Building Fidelity Scaling Curves

A fidelity scaling curve shows how circuit fidelity decays as you
increase circuit size. This is useful for determining the maximum
practical circuit size for a given backend.

In [9]:
est = FidelityEstimator()

# Compare scaling curves for IBM Fez vs Rigetti Ankaa
backends = {
    "ibm_fez": BACKEND_CONFIGS["ibm_fez"],
    "rigetti_ankaa": BACKEND_CONFIGS["rigetti_ankaa"],
    "ionq_aria": BACKEND_CONFIGS["ionq_aria"],
}

depths = [5, 10, 20, 50, 100, 200]

print(f"{'Depth':>6} ", end="")
for name in backends:
    print(f"| {name:>16} ", end="")
print()
print("-" * 60)

for depth in depths:
    print(f"{depth:>6} ", end="")
    for name, spec in backends.items():
        f = est.estimate_depth_limited_fidelity(
            n_two_qubit_gates=depth,
            avg_two_qubit_error=spec.median_cx_error,
            n_one_qubit_gates=depth * 2,
            avg_one_qubit_error=spec.median_cx_error * 0.1,
        )
        print(f"| {f:>16.4f} ", end="")
    print()

 Depth |          ibm_fez |    rigetti_ankaa |        ionq_aria 
------------------------------------------------------------
     5 |           0.9704 |           0.8860 |           0.9762 
    10 |           0.9416 |           0.7850 |           0.9531 
    20 |           0.8867 |           0.6162 |           0.9083 
    50 |           0.7403 |           0.2981 |           0.7863 
   100 |           0.5481 |           0.0889 |           0.6183 
   200 |           0.3004 |           0.0079 |           0.3823 


## 8. Finding the Maximum Useful Circuit Depth

Given a backend and a fidelity threshold, we can binary-search for the
maximum number of 2Q gates before fidelity drops below the threshold.

In [10]:
def max_useful_depth(cx_error, threshold=0.5):
    """Find max 2Q gate count before fidelity drops below threshold."""
    est = FidelityEstimator()
    lo, hi = 0, 10000
    while lo < hi:
        mid = (lo + hi + 1) // 2
        f = est.estimate_depth_limited_fidelity(
            n_two_qubit_gates=mid,
            avg_two_qubit_error=cx_error,
        )
        if f >= threshold:
            lo = mid
        else:
            hi = mid - 1
    return lo


print("Maximum useful 2Q gate count (fidelity >= 0.5):")
print(f"{'Backend':<18} {'CX error':>10} {'Max 2Q gates':>14}")
print("-" * 44)
for name, spec in sorted(BACKEND_CONFIGS.items()):
    max_d = max_useful_depth(spec.median_cx_error)
    print(f"{name:<18} {spec.median_cx_error:>10.4f} {max_d:>14}")

Maximum useful 2Q gate count (fidelity >= 0.5):
Backend              CX error   Max 2Q gates
--------------------------------------------
ibm_fez                0.0050            138
ibm_marrakesh          0.0055            125
ibm_torino             0.0060            115
ionq_aria              0.0040            172
ionq_forte             0.0030            230
iqm_emerald            0.0080             86
iqm_garnet             0.0150             45
quantinuum_h2          0.0010            692
rigetti_ankaa          0.0200             34


## 9. Confidence in Estimates: When to Trust Them

Fidelity estimates are useful but have limitations:

**Trust the estimate when:**
- Calibration data is fresh (< 2 hours old)
- Circuit uses standard gates that appear in calibration data
- Circuit depth is moderate (fidelity > 0.3)

**Be sceptical when:**
- Calibration data is stale (> 12 hours)
- Circuit uses custom gates not in calibration
- Very deep circuits (fidelity < 0.1) -- noise is harder to model accurately
- Crosstalk-heavy circuits (many parallel 2Q gates) -- our model is per-qubit

**The estimate is always optimistic** because it ignores:
- Crosstalk between parallel gates
- Leakage to non-computational states
- Calibration drift during execution
- SPAM (state preparation) errors beyond readout

In [11]:
# Demonstrate the gap between quick estimate and full estimate
est = FidelityEstimator()
spec = BACKEND_CONFIGS["ibm_fez"]

nm = build_noise_model(
    gate_err=spec.median_cx_error,
    readout_err=spec.median_readout_error,
    t1_us=spec.t1_us,
    t2_us=spec.t2_us,
    n_qubits=10,
)

# 10-qubit chain
chain_gates = [("h", (0,))] + [("cx", (i, i + 1)) for i in range(9)]
chain = FECircuit(gates=chain_gates, n_qubits=10, measurements=frozenset(range(10)))

quick = est.estimate_depth_limited_fidelity(
    n_two_qubit_gates=9,
    avg_two_qubit_error=spec.median_cx_error,
    n_one_qubit_gates=1,
    avg_one_qubit_error=spec.median_cx_error * 0.1,
)
full = est.estimate(chain, nm)

print(f"10-qubit chain on IBM Fez:")
print(f"  Quick estimate (gate errors only): {quick:.4f}")
print(f"  Full estimate (+ readout + T1/T2): {full:.4f}")
print(f"  Gap: {quick - full:.4f}")
print(f"\nThe full estimate is {((quick - full) / quick) * 100:.1f}% lower.")
print("Use the full estimate for go/no-go decisions.")
print("Use the quick estimate for rough planning and comparison.")

10-qubit chain on IBM Fez:
  Quick estimate (gate errors only): 0.9554
  Full estimate (+ readout + T1/T2): 0.8037
  Gap: 0.1517

The full estimate is 15.9% lower.
Use the full estimate for go/no-go decisions.
Use the quick estimate for rough planning and comparison.


## 10. Empty and Trivial Circuits

Edge cases: the estimator handles empty circuits (fidelity = 1.0)
and measurement-only circuits correctly.

In [12]:
est = FidelityEstimator()
nm = build_noise_model(0.005, 0.01, 300.0, 150.0, 3)

# Empty circuit
empty = FECircuit(gates=[], n_qubits=3, measurements=frozenset({0, 1, 2}))
print(f"Empty circuit fidelity: {est.estimate(empty, nm):.4f}  (always 1.0)")

# Single gate
single = FECircuit(gates=[("h", (0,))], n_qubits=3, measurements=frozenset({0}))
print(f"Single H gate, 1 measurement: {est.estimate(single, nm):.4f}")

# Same gate, but measure all 3 qubits
single_all = FECircuit(gates=[("h", (0,))], n_qubits=3)
print(f"Single H gate, 3 measurements: {est.estimate(single_all, nm):.4f}")
print("  (lower because more readout errors accumulate)")

Empty circuit fidelity: 1.0000  (always 1.0)
Single H gate, 1 measurement: 0.9891
Single H gate, 3 measurements: 0.9694
  (lower because more readout errors accumulate)


## Summary

Key takeaways:

| Method | Speed | Accuracy | Use case |
|--------|-------|----------|----------|
| `estimate_depth_limited_fidelity()` | Instant | Upper bound | Planning, rough comparison |
| `estimate()` with noise model | Fast | Good | Go/no-go before hardware |

**Rules of thumb:**
- Fidelity > 0.5: circuit should produce usable results
- Fidelity 0.1--0.5: might work with error mitigation
- Fidelity < 0.1: too deep for the hardware -- simplify the circuit
- 2Q gates dominate fidelity loss -- minimise them first